# Cache-Augmented Generation: Key Scope, Reuse, and Invalidation

| Field | Value |
|---|---|
| Stage | Performance and caching |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Prompt/context reuse and semantic answer caching solve different problems and need different correctness boundaries.

## 30-Second Summary

A versioned context cache safely reuses a prepared policy prefix; a semantic answer cache demonstrates a stale date-scoped answer and is rejected by an as-of/version guard.

## Why This Matters

Caching can reduce latency and cost, but a fast stale answer is a correctness incident. Cache identity must include every input that changes meaning.

## Scope

| Covers | Does not cover |
|---|---|
| Context/prompt cache, answer-cache contrast, version/as-of keys, invalidation, hit metrics | Provider billing details, distributed eviction, embedding-based ANN cache |


## Mental Model

```text
immutable prepared context + version -> context cache
query meaning + policy/version/as-of -> answer cache (higher risk)
source change -> invalidate/re-key
```


In [1]:
from hashlib import sha256

policies = {
    "v1": {"effective": "2025-01-01", "days": 30, "text": "Logs are retained for 30 days."},
    "v2": {"effective": "2026-10-01", "days": 45, "text": "Logs are retained for 45 days."},
}
context_cache = {}
answer_cache = {}

def digest(*parts: str) -> str:
    return sha256("|".join(parts).encode()).hexdigest()[:12]


## How It Works

The context cache keys prepared immutable source content by knowledge-base version and prompt template. Answer caching additionally needs query semantics, authorization, model/prompt version, and temporal scope.


## Baseline

A semantic answer cache keyed only by normalized wording returns the v1 answer even after the v2 policy becomes effective.


In [2]:
semantic_key = "how long are logs retained"
answer_cache[semantic_key] = {"answer": "30 days", "policy_version": "v1"}
stale_hit = answer_cache[semantic_key]
stale_hit


{'answer': '30 days', 'policy_version': 'v1'}

## Technique Implementation

A safer context key incorporates source version and template. An answer guard requires the active policy version and as-of date to match before reuse.


In [3]:
def prepared_context(version: str, template_version: str = "prompt-v1") -> tuple[str, bool]:
    key = digest("context", version, template_version, policies[version]["text"])
    hit = key in context_cache
    context_cache.setdefault(key, policies[version]["text"])
    return context_cache[key], hit

def reusable_answer(entry: dict, active_version: str, as_of: str) -> bool:
    return entry["policy_version"] == active_version and entry.get("as_of") == as_of

context_first, first_hit = prepared_context("v2")
context_second, second_hit = prepared_context("v2")
guarded_reuse = reusable_answer(stale_hit, "v2", "2026-10-02")
first_hit, second_hit, guarded_reuse


(False, True, False)

## Controlled Experiment

We require a miss-then-hit for unchanged context and rejection of the stale answer after the policy boundary.


In [4]:
fresh_answer = {"answer": "45 days", "policy_version": "v2", "as_of": "2026-10-02", "source": "policy-v2"}
final = stale_hit if guarded_reuse else fresh_answer
metrics = {"context_hits": int(first_hit) + int(second_hit), "context_requests": 2, "stale_answer_reused": guarded_reuse, "final": final}
metrics


{'context_hits': 1,
 'context_requests': 2,
 'stale_answer_reused': False,
 'final': {'answer': '45 days',
  'policy_version': 'v2',
  'as_of': '2026-10-02',
  'source': 'policy-v2'}}

## Evaluation

The prepared context produces a deterministic **miss then hit**. The incomplete semantic key is rejected, so the final answer is **45 days [policy-v2]** for the new effective period.


In [5]:
assert first_hit is False and second_hit is True and context_first == context_second
assert not metrics["stale_answer_reused"]
assert metrics["final"]["answer"] == "45 days" and metrics["final"]["source"] == "policy-v2"
print("Cache-RAG checks passed.")


Cache-RAG checks passed.


## Decision Guide

| Reuse target | Risk/requirement |
|---|---|
| Tokenized/prefilled stable prefix | Versioned context key |
| Exact deterministic tool result | Exact inputs and TTL |
| Semantic final answer | High risk; scope/version/auth guards |
| Volatile or personalized claim | Bypass or very short TTL |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Stale answer | Source version absent | Version key/event invalidation |
| Cross-user leak | Auth scope absent | Tenant/user permissions in key |
| Low hit rate | Over-specific unstable key | Separate stable context from query |
| False semantic hit | Similar wording, different scope | As-of/entity/intent guards |


## Production Notes

### Observability
Track cache layer, key schema version, hit/miss, age, source version, invalidation reason, and correctness samples.

### Safety and Guardrails
Include authorization scope; never share personalized answers across principals.

### Latency and Cost
Measure end-to-end savings after lookup, serialization, and validation overhead.


## Practice

Add tenant ID and prompt version to the answer key, then test that either change forces a miss.

## Recall

Toggle - Recall: What is safer to cache?
Stable prepared context is generally safer than a final semantic answer.

Toggle - Recall: What invalidates cached RAG state?
Source, prompt/model, authorization, temporal scope, or answer-policy changes.

## Sources

- [OpenAI prompt caching](https://platform.openai.com/docs/guides/prompt-caching)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for version and temporal invalidation behavior | Add tenant keys, TTL metrics, and concurrency tests |
